In [11]:
import pandas as pd
import numpy as np
import time
import os
import sqlite3

# GDP2011/2017

In [19]:
y_end = 2027 #year_end is 2 years from now
y_base = 2011 #for 2017 pricing, switch to 2011 for 2011 pricing
#y_base = 2011

In [20]:
wb_curr = pd.read_csv(r"C:\Users\Norah\Downloads\API_NY.GDP.MKTP.CD_DS2_en_csv_v2_25118\API_NY.GDP.MKTP.CD_DS2_en_csv_v2_25118.csv", skiprows=4)
c_drop = [c for c in wb_curr.columns if c.startswith("Unnamed")]
wb_curr.drop(columns=['Country Code', 'Indicator Name', 'Indicator Code'], inplace=True)
wb_curr.drop(columns=c_drop, inplace=True)
wb_curr.rename(columns={'Country Name':'Country'}, inplace=True)

# WB growth
wb_growth = pd.read_csv(r"C:\Users\Norah\Downloads\API_NY.GDP.MKTP.KD.ZG_DS2_en_csv_v2_23243\API_NY.GDP.MKTP.KD.ZG_DS2_en_csv_v2_23243.csv", skiprows=4)
c_drop = [c for c in wb_growth.columns if c.startswith("Unnamed")]
wb_growth.drop(columns=['Country Code', 'Indicator Name', 'Indicator Code'], inplace=True)
wb_growth.drop(columns=c_drop, inplace=True)
wb_growth.rename(columns={'Country Name':'Country'}, inplace=True)

#growth rates from IFsHistSeries
conn= sqlite3.connect('C:\IFs\DATA\IFsHistSeries.db')
cursor = conn.cursor()
ifs_gdp = pd.read_sql_query(f"SELECT * FROM [SeriesGDP2011];", conn)
conn.close()
ifs_gdp.drop(columns=['Earliest', 'MostRecent','FIPS_CODE'], inplace=True)
ifs_gdp.sort_values(by=["Country"],inplace=True)
ifs_gdp.reset_index(drop=True,inplace=True)

ifs_growth = ifs_gdp.iloc[:,1:].pct_change(periods=1, axis=1)*100
ifs_growth.insert(0, "Country", ifs_gdp.Country)

# IMF growth & current
imf_forecast = pd.read_csv(r"C:\Users\Norah\Downloads\dataset_2025-10-15T19_35_42.513775053Z_DEFAULT_INTEGRATION_IMF.RES_WEO_9.0.0.csv")

#imf_forecast = imf_forecast.replace('n/a', np.nan)
imf_forecast[['CountryCode', 'Variable', 'Frequency']] = imf_forecast['SERIES_CODE'].str.split('.', expand=True)
imf_forecast.drop(columns=['DATASET','OBS_MEASURE','FREQUENCY','CountryCode', 'Frequency'], inplace=True)	
year_cols = [col for col in imf_forecast.columns if col.isdigit()]
display_cols = ['SERIES_CODE','COUNTRY','INDICATOR','SCALE','Variable'] + year_cols
imf_forecast = imf_forecast[display_cols]
imf_forecast.drop(columns=['SERIES_CODE','SCALE','INDICATOR'], inplace=True)
imf_forecast.rename(columns={"COUNTRY":"Country"}, inplace=True)
imf_growth = imf_forecast[imf_forecast['Variable']=='NGDP_RPCH'].copy(True)
imf_growth.drop(columns=['Variable'], inplace=True)
imf_curr = imf_forecast[imf_forecast['Variable']=='NGDPD'].copy(True)
imf_curr.drop(columns=['Variable'], inplace=True)
imf_curr.rename(columns={"COUNTRY":"Country"}, inplace=True)
imf_growth.rename(columns={"COUNTRY":"Country"}, inplace=True)



<>:15: SyntaxWarning: invalid escape sequence '\I'
<>:15: SyntaxWarning: invalid escape sequence '\I'
C:\Users\Norah\AppData\Local\Temp\ipykernel_416\4192348512.py:15: SyntaxWarning: invalid escape sequence '\I'
  conn= sqlite3.connect('C:\IFs\DATA\IFsHistSeries.db')
C:\Users\Norah\AppData\Local\Temp\ipykernel_416\4192348512.py:23: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  ifs_growth = ifs_gdp.iloc[:,1:].pct_change(periods=1, axis=1)*100


In [21]:
# imf_forecast.columns
imf_forecast[imf_forecast['Variable'].isin(['NGDP_RPCH','NGDPD'])].drop_duplicates('Variable')


,Country,Variable,1980,1981,1982,1983,1984,1985,1986,1987,...,2021,2022,2023,2024,2025,2026,2027,2028,2029,2030
33,Advanced Economies,NGDP_RPCH,1.313000e+00,2.295000e+00,2.360000e-01,3.175000e+00,4.811000e+00,3.687000e+00,3.278000e+00,3.790000e+00,...,6.033000e+00,2.980000e+00,1.727000e+00,1.827000e+00,1.607000e+00,1.634000e+00,1.725000e+00,1.712000e+00,1.596000e+00,1.519000e+00
125,Advanced Economies,NGDPD,8.478682e+12,8.614064e+12,8.553227e+12,8.900939e+12,9.308571e+12,9.800961e+12,1.196912e+13,1.391772e+13,...,5.760952e+13,5.894592e+13,6.231026e+13,6.490112e+13,6.859862e+13,7.220272e+13,7.494823e+13,7.781077e+13,8.067039e+13,8.375024e+13


## Country Concordance

In [22]:
c_table = pd.read_excel(r"C:\Users\Norah\Downloads\GDPPCPPP_Oct24 (1)\GDPPCPPP_Oct24\country_concordance 20231201.xlsx")
c_ifs_new = ifs_gdp.Country.unique()
c_wb = wb_growth.Country.unique()
c_imf = imf_forecast.Country.dropna().unique()
# 
imf_to_ifs = dict(zip(c_table["IMF WEO Countries"], c_table["Country"]))
if np.nan in imf_to_ifs:
    del imf_to_ifs[np.nan]
wb_to_ifs = dict(zip(c_table["World Bank Countries"], c_table["Country"]))
if np.nan in wb_to_ifs:
    del wb_to_ifs[np.nan]


## Wide to Long

In [23]:
wb_curr.Country = wb_curr.Country.replace(wb_to_ifs)
wb_growth.Country = wb_growth.Country.replace(wb_to_ifs)
imf_curr.Country = imf_curr.Country.replace(imf_to_ifs)
imf_growth.Country = imf_growth.Country.replace(imf_to_ifs)

wb_curr_long = wb_curr.melt(id_vars="Country", var_name = "Year", value_name="wb_curr")
wb_curr_long.Year = wb_curr_long.Year.astype(int)
wb_growth_long = wb_growth.melt(id_vars="Country", var_name = "Year", value_name="wb_growth")
wb_growth_long.Year = wb_growth_long.Year.astype(int)
imf_curr_long = imf_curr.melt(id_vars="Country", var_name = "Year", value_name="imf_curr")
imf_curr_long.Year = imf_curr_long.Year.astype(int)
imf_curr_long["imf_curr"] = imf_curr_long["imf_curr"].replace({"--":np.nan})
imf_growth_long = imf_growth.melt(id_vars="Country", var_name = "Year", value_name="imf_growth")
imf_growth_long.Year = imf_growth_long.Year.astype(int)
imf_growth_long["imf_growth"] = imf_growth_long["imf_growth"].replace({"--":np.nan})

ifs_gdp_long = ifs_gdp.melt(id_vars="Country", var_name = "Year", value_name="ifs_gdp")
ifs_gdp_long.ifs_gdp = ifs_gdp_long.ifs_gdp*1000000000
ifs_gdp_long.Year = ifs_gdp_long.Year.astype(int)
ifs_growth_long = ifs_growth.melt(id_vars="Country", var_name = "Year", value_name="ifs_growth")
ifs_growth_long.Year = ifs_growth_long.Year.astype(int)

## Start to calculate new GDP for each country

In [24]:
gdp_frame_list = []
#
for c in c_ifs_new:
    # empty frame
    c_frame_long = pd.DataFrame({"Country":[c]*len(range(1960, y_end+1)), "Year": [i for i in range(1960, y_end+1)], "GDP_curr": np.nan})
    c_frame_long = pd.merge(left=c_frame_long, right=wb_curr_long, on=["Country", "Year"], how="left")
    c_frame_long = pd.merge(left=c_frame_long, right=imf_curr_long, on=["Country", "Year"], how="left")
    c_frame_long = pd.merge(left=c_frame_long, right=ifs_gdp_long, on=["Country", "Year"], how="left")
    # GDP year_base 
    c_frame_long.GDP_curr.fillna(c_frame_long.wb_curr, inplace=True)
    c_frame_long.GDP_curr.fillna(c_frame_long.imf_curr, inplace=True)
    c_frame_long.GDP_curr.fillna(c_frame_long.ifs_gdp, inplace=True)
    
    c_frame_long.drop(columns=["wb_curr", "imf_curr"], inplace=True)
    # GDP growth
    c_frame_long = pd.merge(left=c_frame_long, right=wb_growth_long, on=["Country", "Year"], how="left")
    c_frame_long = pd.merge(left=c_frame_long, right=ifs_growth_long, on=["Country", "Year"], how="left")
    c_frame_long = pd.merge(left=c_frame_long, right=imf_growth_long, on=["Country", "Year"], how="left")
    c_frame_long["Growth"] = c_frame_long.wb_growth.fillna(c_frame_long.imf_growth)
    c_frame_long.Growth = c_frame_long.Growth.fillna(c_frame_long.ifs_growth)
    c_frame_long.drop(columns=["wb_growth", "ifs_growth", "imf_growth"], inplace=True)
    c_frame_long.Growth = c_frame_long.Growth.fillna(method="ffill",axis =0)
    c_frame_long["GDP_new"] = np.nan
    c_frame_long.loc[c_frame_long.Year==y_base, "GDP_new"] = c_frame_long.loc[c_frame_long.Year==y_base, "GDP_curr"].values[0] # fill base year value of GDP_curr to GDP_new
    # before year base 
    for y in range(y_base-1, 1959, -1):
        # print(y)
        if c_frame_long[c_frame_long.Year==y+1].Growth.isna().values[0]:
            break
        else:
            c_frame_long.loc[c_frame_long.Year==y, "GDP_new"] = c_frame_long.loc[c_frame_long.Year==y+1, "GDP_new"].values[0] * 100 / (100 + c_frame_long.loc[c_frame_long.Year==y+1, "Growth"].values[0])  # e.g. GDP_new in 2010 = GDP_new in 2011 * 100 / (Growth in 2011 + 100)
    # after year base 
    for y in range(y_base+1, y_end+1, 1):
        if c_frame_long[c_frame_long.Year==y].Growth.isna().values[0]:
            break
        else:
            c_frame_long.loc[c_frame_long.Year==y, "GDP_new"] = c_frame_long.loc[c_frame_long.Year==y-1, "GDP_new"].values[0] * (100 + c_frame_long.loc[c_frame_long.Year==y, "Growth"].values[0]) / 100   # e.g. GDP_new in 2012 = GDP_new in 2011 * (100 + Growth in 2012) / 100                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                 
    gdp_frame_list.append(c_frame_long)
    # 
gdp_frame = pd.concat(gdp_frame_list)


C:\Users\Norah\AppData\Local\Temp\ipykernel_416\719390420.py:10: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  c_frame_long.GDP_curr.fillna(c_frame_long.wb_curr, inplace=True)
C:\Users\Norah\AppData\Local\Temp\ipykernel_416\719390420.py:11: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a co

## Save GDP to csv

In [25]:
gdp_frame_short = gdp_frame[["Country", "Year", "ifs_gdp", "GDP_new"]].copy()
gdp_frame_short.GDP_new = gdp_frame_short.GDP_new/1000000000
gdp_frame_short.ifs_gdp = gdp_frame_short.ifs_gdp/1000000000
gdp_frame_short.GDP_new = round(gdp_frame_short.GDP_new,5)

first_last = gdp_frame_short.sort_values(['Country','Year']).groupby(gdp_frame_short['Country'])['GDP_new'] \
        .agg(['first', 'last']).reset_index()
first_last = first_last.rename(columns={"first": "Earliest", "last": "MostRecent"})

gdp_frame_short_wide = gdp_frame_short.pivot(index="Country", columns="Year", values="GDP_new").reset_index()
gdp_frame_short_wide = gdp_frame_short_wide.merge(first_last,
                on = 'Country',
                how = 'left')
country = pd.read_excel(r"C:\Users\Norah\Downloads\GDPPCPPP_Oct24 (1)\GDPPCPPP_Oct24\country_concordance 20231201.xlsx", sheet_name='Sheet2')
gdp_frame_short_wide = country.merge(gdp_frame_short_wide,
                    on = 'Country',
                    how = 'left')
print(y_base)
gdp_frame_short_wide

#change year in csv
gdp_frame_short_wide.to_csv(f"GDP{y_base}Updated.csv", index=False)

2011


## TO SQL

In [30]:
y_base = 2021 #for 2017 pricing, switch to 2011 for 2011 pricing

In [31]:
#change csv files when switching to 2011
df = pd.read_csv(f'GDP{y_base}Updated.csv')
year_columns = [col for col in df.columns if str(col).isdigit()]

#sql table; change table name each time you import 
conn = sqlite3.connect(r"C:\IFs\RUNFILES\IFsDataImportCompletedSpecialCases - Copy.db")
cursor = conn.cursor()
table_name = 'SeriesGDP{y_base}'
columns_sql = ['Country VARCHAR(255)', 'FIPS_CODE VARCHAR(255)']
columns_sql += [f'"{col}" DOUBLE(53)' for col in year_columns + ['Earliest', 'MostRecent']]
create_table_sql = f"CREATE TABLE '{table_name}' ({', '.join(columns_sql)})"
cursor.execute(f'DROP TABLE IF EXISTS "{table_name}"')
cursor.execute(create_table_sql)

df.to_sql(table_name, conn, if_exists='replace', index=False)

conn.commit()
conn.close()

# GDPPC_PPP2011/2017

In [26]:
#y_target = 2017
#y_target = 2011
y_target = 2021

## read raw data

In [29]:
# WB PC current USD
wb_curr = pd.read_csv(r"C:\Users\Norah\Downloads\API_NY.GDP.PCAP.CN_DS2_en_csv_v2_22672\API_NY.GDP.PCAP.CN_DS2_en_csv_v2_22672.csv", skiprows=4)
c_drop = [c for c in wb_curr.columns if c.startswith("Unnamed")]
wb_curr.drop(columns=['Country Code', 'Indicator Name', 'Indicator Code'], inplace=True)
wb_curr.drop(columns=c_drop, inplace=True)
wb_curr.rename(columns={'Country Name':'Country'}, inplace=True)
# WB GDP Deflator
wb_deflat = pd.read_csv(r"C:\Users\Norah\Downloads\API_NY.GDP.DEFL.ZS_DS2_en_csv_v2_6548\API_NY.GDP.DEFL.ZS_DS2_en_csv_v2_6548.csv", skiprows=4)
c_drop = [c for c in wb_deflat.columns if c.startswith("Unnamed")]
wb_deflat.drop(columns=['Country Code', 'Indicator Name', 'Indicator Code'], inplace=True)
wb_deflat.drop(columns=c_drop, inplace=True)
wb_deflat.rename(columns={'Country Name':'Country'}, inplace=True)
# WB PPP conversion
wb_ppp_cov = pd.read_csv(r"C:\Users\Norah\Downloads\API_PA.NUS.PPP_DS2_en_csv_v2_5722\API_PA.NUS.PPP_DS2_en_csv_v2_5722.csv", skiprows=4)
c_drop = [c for c in wb_ppp_cov.columns if c.startswith("Unnamed")]
wb_ppp_cov.drop(columns=['Country Code', 'Indicator Name', 'Indicator Code'], inplace=True)
wb_ppp_cov.drop(columns=c_drop, inplace=True)
wb_ppp_cov.rename(columns={'Country Name':'Country'}, inplace=True)

In [31]:
# WB growth
wb_growth = pd.read_csv(r"C:\Users\Norah\Downloads\API_NY.GDP.PCAP.KD.ZG_DS2_en_csv_v2_25015\API_NY.GDP.PCAP.KD.ZG_DS2_en_csv_v2_25015.csv", skiprows=4)
c_drop = [c for c in wb_growth.columns if c.startswith("Unnamed")]
wb_growth.drop(columns=['Country Code', 'Indicator Name', 'Indicator Code'], inplace=True)
wb_growth.drop(columns=c_drop, inplace=True)
wb_growth.rename(columns={'Country Name':'Country'}, inplace=True)
# IFs pcppp & growth
conn= sqlite3.connect(r'C:\IFs\DATA\IFsHistSeries.db')
cursor = conn.cursor()
#ifs_pcppp = pd.read_sql_query(f"SELECT * FROM [SeriesGDP{y_target}PCPPP];", conn)
ifs_pcppp = pd.read_sql_query(f"SELECT * FROM [SeriesGDP2017PCPPP];", conn)
conn.close()
ifs_pcppp.drop(columns=["FIPS_CODE", "Earliest", "MostRecent"], inplace=True)
ifs_pcppp = ifs_pcppp.sort_values(by=["Country"]).reset_index(drop=True)
ifs_pcppp_growth = ifs_pcppp.iloc[:,1:].pct_change(periods=1, axis=1)*100
ifs_pcppp_growth.insert(0, "Country", ifs_pcppp.Country)

C:\Users\Norah\AppData\Local\Temp\ipykernel_2972\971980851.py:15: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  ifs_pcppp_growth = ifs_pcppp.iloc[:,1:].pct_change(periods=1, axis=1)*100


## Country Concordance

In [32]:
c_table = pd.read_excel(r"C:\Users\Norah\Downloads\GDPPCPPP_Oct24 (1)\GDPPCPPP_Oct24\country_concordance 20231201.xlsx")
c_ifs_new = ifs_pcppp.Country.unique()
c_wb = list(set(wb_growth.Country.unique()) | set(wb_deflat.Country.unique()) | set(wb_ppp_cov.Country.unique()) | set(wb_curr.Country.unique()))
c_wb.sort()
# 
wb_to_ifs = dict(zip(c_table["World Bank Countries"], c_table["Country"]))
if np.nan in wb_to_ifs:
    del wb_to_ifs[np.nan]

wb_curr.Country = wb_curr.Country.replace(wb_to_ifs)
wb_deflat.Country = wb_deflat.Country.replace(wb_to_ifs)
wb_ppp_cov.Country = wb_ppp_cov.Country.replace(wb_to_ifs)
wb_growth.Country = wb_growth.Country.replace(wb_to_ifs)

## Wide to Long

In [33]:
wb_curr_long = wb_curr.melt(id_vars="Country", var_name = "Year", value_name="wb_curr")
wb_curr_long.Year = wb_curr_long.Year.astype(int)
#
wb_deflat_long = wb_deflat.melt(id_vars="Country", var_name = "Year", value_name="wb_deflat")
wb_deflat_long.Year = wb_deflat_long.Year.astype(int)
#
wb_ppp_cov_long = wb_ppp_cov.melt(id_vars="Country", var_name = "Year", value_name="wb_ppp_cov")
wb_ppp_cov_long.Year = wb_ppp_cov_long.Year.astype(int)
#
wb_growth_long = wb_growth.melt(id_vars="Country", var_name = "Year", value_name="wb_growth")
wb_growth_long.Year = wb_growth_long.Year.astype(int)
#
ifs_pcppp_long = ifs_pcppp.melt(id_vars="Country", var_name = "Year", value_name="ifs_pcppp")
ifs_pcppp_long.Year = ifs_pcppp_long.Year.astype(int)
#
ifs_pcppp_growth_long = ifs_pcppp_growth.melt(id_vars="Country", var_name = "Year", value_name="ifs_pcppp_growth")
ifs_pcppp_growth_long.Year = ifs_pcppp_growth_long.Year.astype(int)

## PC_Growth

In [34]:
# GDP PC growth with full coverage
pc_growth_long = pd.merge(left=ifs_pcppp_growth_long, right = wb_growth_long, on=["Country", "Year"], how="outer")
pc_growth_long = pc_growth_long[pc_growth_long.Country.isin(c_ifs_new)].reset_index(drop=True)
assert len(pc_growth_long.Country.unique())==188
pc_growth_long["pc_growth"] =  pc_growth_long.wb_growth.fillna(pc_growth_long.ifs_pcppp_growth)
pc_growth_long = pc_growth_long.dropna(subset=["pc_growth"])
pc_growth_long = pc_growth_long[["Country", "Year", "pc_growth"]].sort_values(by=["Country", "Year"]).reset_index(drop=True)
pc_growth_long

,Country,Year,pc_growth
0,Afghanistan,1961,-0.963257
1,Afghanistan,1962,-0.217568
2,Afghanistan,1963,-0.231212
3,Afghanistan,1964,-0.033708
4,Afghanistan,1965,-0.059715
...,...,...,...
11620,Zimbabwe,2020,-9.333954
11621,Zimbabwe,2021,6.611933
11622,Zimbabwe,2022,4.343582
11623,Zimbabwe,2023,3.584903


## Constant_PCPPP for most recent year for each country

In [35]:
# use y_target = 2017 to verify this method
#y_target = 2017
dt_deflat_target = wb_deflat_long[(wb_deflat_long.Country.isin(c_ifs_new))&(wb_deflat_long.Year==y_target)].dropna()
dt_deflat_target = dt_deflat_target[["Country", "wb_deflat"]]
dt_deflat_target.columns=["Country", f"{y_target}_deflat"]
dt_deflat_target
#
dt_ppp_cov_target = wb_ppp_cov_long[(wb_ppp_cov_long.Country.isin(c_ifs_new))&(wb_ppp_cov_long.Year==y_target)].dropna()
dt_ppp_cov_target = dt_ppp_cov_target[["Country", "wb_ppp_cov"]]
dt_ppp_cov_target.columns=["Country", f"{y_target}_ppp_cov"]
#
wb_curr_long = wb_curr_long[wb_curr_long.Country.isin(c_ifs_new)].dropna()
wb_deflat_long = wb_deflat_long[wb_deflat_long.Country.isin(c_ifs_new)].dropna()
wb_curr_deflat_long = pd.merge(left=wb_curr_long, right=wb_deflat_long, on=["Country","Year"],how="inner")
wb_curr_deflat_long
wb_curr_deflat_long_ymax = wb_curr_deflat_long.groupby(["Country"]).Year.max().reset_index() # find max year for each country
wb_curr_deflat_long_ymax
wb_curr_deflat_long_ymax = pd.merge(left=wb_curr_deflat_long_ymax, right=wb_curr_deflat_long, on=["Country","Year"],how="left") # wb_curr and wb_deflat for each country in the most recent year
wb_curr_deflat_long_ymax
#
dt_deflat_ymax = pd.merge(left=wb_curr_deflat_long_ymax, right=dt_deflat_target, on =["Country"], how = "inner")
dt_deflat_ymax = pd.merge(left=dt_deflat_ymax, right=dt_ppp_cov_target, on =["Country"], how = "inner")
# LCU per international$
dt_deflat_ymax[f"{y_target}_const_pcppp"] = (dt_deflat_ymax.wb_curr * (dt_deflat_ymax[f"{y_target}_deflat"]/dt_deflat_ymax.wb_deflat))/dt_deflat_ymax[f"{y_target}_ppp_cov"]
dt_deflat_ymax.head()

,Country,Year,wb_curr,wb_deflat,2021_deflat,2021_ppp_cov,2021_const_pcppp
0,Afghanistan,2023,3.258757e+04,127.912184,113.593791,14.587942,1983.812620
1,Albania,2024,9.320075e+05,123.780604,103.444391,41.165376,18920.894264
2,Algeria,2024,7.548779e+05,392.035022,310.866327,38.763100,15442.124446
3,Angola,2024,2.141241e+06,5045.663717,3012.792411,174.090515,7344.145379
4,Argentina,2024,1.267603e+07,82492.351478,6700.263541,38.783333,26547.050343


## Get the most recent year and const_pcppp for all the countries

In [36]:
c_pcppp_ifs = [] 
for c in c_ifs_new:
    if c not in dt_deflat_ymax.Country.unique():
        c_pcppp_ifs.append(c)
ifs_pcppp_long_subset = ifs_pcppp_long[ifs_pcppp_long.Country.isin(c_pcppp_ifs)].sort_values(by=["Country", "Year"]).dropna()
ifs_pcppp_long_subset = ifs_pcppp_long_subset.groupby("Country").tail(1)
ifs_pcppp_long_subset.columns = ["Country", "Year", f"{y_target}_const_pcppp"]
ifs_pcppp_long_subset
#
dt_const_pcppp = pd.concat([ifs_pcppp_long_subset, dt_deflat_ymax[["Country", "Year", f"{y_target}_const_pcppp"]]])
dt_const_pcppp = dt_const_pcppp.sort_values(by=["Country", "Year"]).reset_index(drop=True)
dt_const_pcppp

,Country,Year,2021_const_pcppp
0,Afghanistan,2023,1983.812620
1,Albania,2024,18920.894264
2,Algeria,2024,15442.124446
3,Angola,2024,7344.145379
4,Argentina,2024,26547.050343
...,...,...,...
183,"Venezuela, Bolivarian Republic",2023,9467.942380
184,Viet Nam,2024,14415.215953
185,Yemen,2023,1583.793700
186,Zambia,2024,3716.037353


## Start to calculate PCPPP for each country and year

In [37]:
y_end = 2024
pcppp_frame_list = []
#
for c in c_ifs_new:
    # print(c)
    #
    c_list = []
    y_list = []
    pcppp_list = []
    # y_mid to 1960 or erliest
    y_mid =  dt_const_pcppp[dt_const_pcppp.Country==c]["Year"].values[0] # the most recent year for the country
    # print(y_mid)
    y_earliest = pc_growth_long[pc_growth_long.Country==c].Year.min()
    # print(y_earliest)
    pcppp_y = dt_const_pcppp[dt_const_pcppp.Country==c][f"{y_target}_const_pcppp"].values[0] # the const_pcppp in the most recent year for the country
    #
    for y in range(y_mid, y_earliest-1, -1):
        try:
            growth_y = pc_growth_long[(pc_growth_long.Country==c) & (pc_growth_long.Year==y)].pc_growth.values[0] # e.g. growth_rate in 2022 for the country
        except:
            growth_y = 0
        pcppp_ymin1 = 100* pcppp_y / (100+ growth_y) # pcppp in 2021 for the country
        #
        pcppp_list.append(pcppp_ymin1) 
        c_list.append(c)
        y_list.append(y-1)
        pcppp_y = pcppp_ymin1
    # # y_mid to y_end
    y_mid =  dt_const_pcppp[dt_const_pcppp.Country==c]["Year"].values[0]
    pcppp_y = dt_const_pcppp[dt_const_pcppp.Country==c][f"{y_target}_const_pcppp"].values[0]
    for y in range(y_mid, y_end):
        try:
            growth_yplus1 = pc_growth_long[(pc_growth_long.Country==c) & (pc_growth_long.Year == y+1 )].pc_growth.values[0] # e.g. growth_rate in 2022 for the country
        except:
            growth_yplus1 = 0
        pcppp_yplus1 = pcppp_y * (100+ growth_yplus1)/100 # pcppp in 2022 for the country
        #
        pcppp_list.append(pcppp_yplus1)
        c_list.append(c)
        y_list.append(y+1)
        pcppp_y = pcppp_yplus1  
    ###
    dt_pcppp_c = pd.DataFrame({"Country": c_list, "Year": y_list, f"{y_target}_const_pcppp": pcppp_list})
    pcppp_frame_list.append(dt_pcppp_c)


## Get the final GDPPCPPP

In [38]:
dt_pcppp_master = pd.concat(pcppp_frame_list, ignore_index=True)
dt_pcppp_master = pd.concat([dt_pcppp_master, dt_const_pcppp], ignore_index=True) # merge with the most recent year value
dt_pcppp_master = dt_pcppp_master.sort_values(by=["Country", "Year"]).reset_index(drop=True)
dt_pcppp_master
dt_const_pcppp.sort_values(by = ['Year'])
dt_pcppp_master[f"{y_target}_const_pcppp"] = round(dt_pcppp_master[f"{y_target}_const_pcppp"],5)
dt_pcppp_master = dt_pcppp_master.drop_duplicates(ignore_index=True)

first_last = dt_pcppp_master.sort_values(['Country','Year']).groupby(dt_pcppp_master['Country'])[f"{y_target}_const_pcppp"] \
        .agg(['first', 'last']).reset_index()
first_last = first_last.rename(columns={"first": "Earliest", "last": "MostRecent"})



dt_pcppp_master_wide = dt_pcppp_master.reset_index().pivot_table(index="Country", columns="Year", values=f"{y_target}_const_pcppp").reset_index()
dt_pcppp_master_wide = dt_pcppp_master_wide.merge(first_last,
                on = 'Country',
                how = 'left')
country = pd.read_excel(r'C:\Users\Norah\Downloads\GDPPCPPP_Oct24 (1)\GDPPCPPP_Oct24\country_concordance 20231201.xlsx', sheet_name='Sheet2')
dt_pcppp_master_wide = country.merge(dt_pcppp_master_wide,
                    on = 'Country',
                    how = 'left')
dt_pcppp_master_wide
dt_pcppp_master_wide.to_csv(f"2021PCPPP1.csv", index=False)



In [39]:
#change csv files when switching to 2011
df = pd.read_csv('2021PCPPP1.csv')
year_columns = [col for col in df.columns if str(col).isdigit()]

#sql table; change table name each time you import 
conn = sqlite3.connect(r'C:\IFs\RUNFILES\IFsDataImport - Copy (7) - Copy - Copy.db')
cursor = conn.cursor()
table_name = 'SeriesGDP2021PCPPP'
columns_sql = ['Country VARCHAR(255)', 'FIPS_CODE VARCHAR(255)']
columns_sql += [f'"{col}" DOUBLE(53)' for col in year_columns + ['Earliest', 'MostRecent']]
create_table_sql = f"CREATE TABLE '{table_name}' ({', '.join(columns_sql)})"
cursor.execute(f'DROP TABLE IF EXISTS "{table_name}"')
cursor.execute(create_table_sql)

df.to_sql(table_name, conn, if_exists='append', index=False)
1

conn.commit()
conn.close()

# IFs Import

In [ ]:
dd = pd.read_excel('DataDict.xlsx')
conn = sqlite3.connect('IFsDataImport.db')
cursor = conn.cursor()

dd.to_sql(name=f'DataDict', con=conn, if_exists="replace", index=False)
conn.close()